# 3. Train a model and register it to Unity Catalog
Predict fare from trip distance. We keep the model trivial on purpose — the point is the MLflow + UC workflow, not the algorithm.

**Concepts:** MLflow tracking, experiments, UC registered models, versioning.

In [ ]:
dbutils.widgets.text("catalog", "main")
dbutils.widgets.text("schema", "ml_workshop")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
model_name = f"{catalog}.{schema}.taxi_fare"

In [ ]:
import mlflow
from sklearn.linear_model import LinearRegression

# Tell MLflow to register models in Unity Catalog (not the workspace registry).
mlflow.set_registry_uri("databricks-uc")

pdf = spark.table("samples.nyctaxi.trips").toPandas()
X = pdf[["trip_distance"]]
y = pdf["fare_amount"]

In [ ]:
# autolog captures params, metrics, and the model artifact automatically.
mlflow.sklearn.autolog()

with mlflow.start_run() as run:
    model = LinearRegression().fit(X, y)
    # Enrichment: swap LinearRegression for XGBoost/LightGBM, add a train/test
    # split and metrics, or run a hyperparameter sweep — the flow is identical.

# Register this run's model as a new version in Unity Catalog.
mlflow.register_model(f"runs:/{run.info.run_id}/model", model_name)

> **Expand here:** set an alias (`@champion`) for the version to promote, compare runs in the Experiments UI, or add Model Serving for real-time inference.